# CUAD evaluation with vLLM (Kaggle)

Runs the baselines and the fine-tuned model on the CUAD test split and scores them.

**Notebook settings:** Accelerator **GPU T4 x2**, Internet **On**. For the fine-tuned tasks, **Add Input →
the training notebook's output**; the notebook finds it automatically.

**Tasks** (put the ones to run this session in `TASKS`; each prints its own scores). They run in
the order below whatever order you list them in, and a failed task is reported and skipped so it
never blocks the ones after it:
| Task | What | Rough T4 time |
|---|---|---|
| `select_checkpoint` | If an earlier checkpoint had lower val loss than the final one, scores both on **val** and sets `BEST` to the winner | ~30 min each |
| `test_finetuned` | Score the chosen adapter (`BEST`, default the final one) on **test** | ~1.2 h |
| `zeroshot_4b` | Qwen3-4B-Instruct, no fine-tuning | ~1.2 h |
| `rag_4b` | Hybrid retrieval + Qwen3-4B-Instruct | ~2–2.5 h |
| `zeroshot_8b` | Qwen3-8B, no fine-tuning (split over both T4s) | ~1.2 h |
| `rag_8b` | Hybrid retrieval + Qwen3-8B; too slow for free T4s (~5–6 h), prefer `rag_4b` | ~5–6 h |

Times are estimates. Set `LIMIT = 3` for a quick smoke test first. Responses are cached in
`outputs/<run>/responses.jsonl`. To resume after a failed or cut-off run, attach that run's output
as an input: its caches are copied in automatically and only unfinished requests are sent.

If Qwen3-8B produces garbage in fp16, swap `Qwen/Qwen3-8B` for `Qwen/Qwen3-8B-AWQ` in the serve call.

In [ ]:
REPO_URL = "https://github.com/lakshy-606/cuad-qlora-vllm.git"
TASKS = ["select_checkpoint", "test_finetuned"]  # see the table above for all tasks
BEST = "adapter"  # the final adapter; select_checkpoint replaces this with the best on val
LIMIT = None  # e.g. 3 contracts for a smoke test

In [ ]:
def sh(cmd):
    """Run a shell command, streaming its output; raise if it fails."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
%cd /kaggle/working
sh(f"rm -rf cuad-qlora-vllm && git clone -q {REPO_URL}")
%cd /kaggle/working/cuad-qlora-vllm
sh("pip install -q vllm")
sh("pip install -q -e .")
if "rag_4b" in TASKS or "rag_8b" in TASKS:
    sh("pip install -q rank-bm25 faiss-cpu sentence-transformers")
sh("python scripts/prepare_data.py")

In [ ]:
import glob
import json
from pathlib import Path

# The training notebook's output, attached with Add Input (found automatically).
_found = glob.glob("/kaggle/input/**/qwen3_4b_cuad_lora/train_summary.json", recursive=True)
TRAIN_OUTPUT = str(Path(_found[0]).parent) if _found else None
print("Training output:", TRAIN_OUTPUT)

# Resume: if an earlier run of this notebook is attached as an input, copy its response caches so
# requests it already finished aren't sent again (the cache key includes the model name).
import shutil

for _src in glob.glob("/kaggle/input/**/cuad-qlora-vllm/outputs/*/responses.jsonl", recursive=True):
    _dst = Path("outputs") / Path(_src).parent.name / "responses.jsonl"
    if not _dst.exists():
        _dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(_src, _dst)
        print(f"Resuming from cached responses: {_dst} ({sum(1 for _ in open(_dst))} responses)")

BASE = "Qwen/Qwen3-4B-Instruct-2507"
# T4 has no bf16; both GPUs share each model (tensor parallel) for memory and speed.
T4 = "--dtype half --tensor-parallel-size 2 --max-model-len 12288 --gpu-memory-utilization 0.90"


_served = None


def serve(model, extra=""):
    """Start vLLM for this model, unless it is already being served with the same settings."""
    global _served
    if _served == (model, extra):
        return
    sh('if [ -f outputs/vllm.pid ]; then kill "$(cat outputs/vllm.pid)" 2>/dev/null; sleep 15; fi')
    _served = None
    sh(f"scripts/start_vllm.sh {model} {T4} {extra}")
    _served = (model, extra)


FAILED = []


def run_task(name, fn):
    """Run one task if it was requested; report a failure and carry on with the rest."""
    if name not in TASKS:
        return
    print(f"\n===== {name} =====")
    try:
        fn()
    except Exception as e:
        FAILED.append(name)
        print(f"!! {name} failed: {e}. Continuing with the remaining tasks.")


def predict(config, *args):
    limit = f" --limit {LIMIT}" if LIMIT else ""
    sh(f"python scripts/predict.py --config {config} {' '.join(args)}{limit}")


def lora_args(adapters):
    mods = " ".join(f"{name}={path}" for name, path in adapters.items())
    return f"--enable-lora --max-lora-rank 16 --max-loras 1 --lora-modules {mods}"


def candidates():
    """The final adapter, plus the lowest-val-loss checkpoint if that's an earlier step (the final
    step's checkpoint has the same weights as the adapter, so scoring it again would waste quota)."""
    assert TRAIN_OUTPUT, "attach the training notebook's output with Add Input"
    out = Path(TRAIN_OUTPUT)
    found = {"adapter": out / "adapter"}
    summary = json.loads((out / "train_summary.json").read_text())
    if summary["eval_losses"]:
        best = min(summary["eval_losses"], key=lambda e: e["eval_loss"])
        ckpt = out / "checkpoints" / f"checkpoint-{best['step']}"
        if best["step"] < summary["global_step"] and ckpt.exists():
            found[ckpt.name] = ckpt
    return found

In [ ]:
# Checkpoint selection uses the val split only; the test split is touched once, by test_finetuned.
def select_checkpoint():
    global BEST
    adapters = candidates()
    print("candidates:", list(adapters))
    serve(BASE, lora_args(adapters))
    runs = []
    for name in adapters:
        run = f"ft_val_{name}"
        predict("configs/finetuned_qwen3_4b.yaml", "--split val", f"--model {name}", f"--run-name {run}")
        runs.append(f"outputs/{run}")
    sh(f"python scripts/evaluate.py {' '.join(runs)}")
    scores = {
        name: json.loads(Path(f"outputs/ft_val_{name}/metrics.json").read_text())["coverage_f1"]
        for name in adapters
    }
    BEST = max(scores, key=scores.get)
    print(f"Best on val: {BEST} (coverage F1 {scores[BEST]:.3f}); test_finetuned will use it")


def test_finetuned():
    # Same server as select_checkpoint (all candidates loaded), so no restart between the two.
    serve(BASE, lora_args(candidates()))
    predict("configs/finetuned_qwen3_4b.yaml", f"--model {BEST}")
    sh("python scripts/evaluate.py outputs/finetuned_qwen3_4b")


run_task("select_checkpoint", select_checkpoint)
run_task("test_finetuned", test_finetuned)

In [ ]:
def zeroshot_4b():
    serve(BASE)
    predict("configs/zeroshot_qwen3_4b.yaml")
    sh("python scripts/evaluate.py outputs/zeroshot_qwen3_4b")


def rag_4b():
    serve(BASE)
    predict("configs/rag_qwen3_4b.yaml")
    sh("python scripts/evaluate.py outputs/rag_qwen3_4b")


def zeroshot_8b():
    serve("Qwen/Qwen3-8B")
    predict("configs/zeroshot_qwen3_8b.yaml")
    sh("python scripts/evaluate.py outputs/zeroshot_qwen3_8b")


def rag_8b():
    serve("Qwen/Qwen3-8B")
    predict("configs/rag_qwen3_8b.yaml")
    sh("python scripts/evaluate.py outputs/rag_qwen3_8b")


run_task("zeroshot_4b", zeroshot_4b)
run_task("rag_4b", rag_4b)
run_task("zeroshot_8b", zeroshot_8b)
run_task("rag_8b", rag_8b)

In [ ]:
# Scoreboard for every test run finished in this session, next to the published ContractEval numbers.
done = [str(p.parent) for p in Path("outputs").glob("*/predictions.jsonl") if not p.parent.name.startswith("ft_val_")]
if done:
    sh(f"python scripts/evaluate.py {' '.join(sorted(done))}")
if FAILED:
    print("Failed tasks:", FAILED)
sh('if [ -f outputs/vllm.pid ]; then kill "$(cat outputs/vllm.pid)" 2>/dev/null; fi')
sh("rm -rf data/raw")